# 🗂️ Hands-On: Web Server Caching dengan Nginx
## Browser Cache vs Proxy Cache — Tanpa Python Backend

**Tujuan Pembelajaran**
Setelah menyelesaikan latihan ini, mahasiswa mampu:
- Membedakan **browser cache** (disimpan di klien) vs **proxy cache** (disimpan di server)
- Membaca dan menginterpretasikan header HTTP: `Cache-Control`, `ETag`, `X-Cache-Status`
- Mengamati siklus **MISS → HIT → EXPIRED** pada nginx proxy cache
- Membuktikan efek caching menggunakan `curl` dan Python `requests`

---

## Prasyarat

Hanya perlu **nginx yang sudah berjalan**. Tidak ada Python backend.

**Aktivasi konfigurasi demo** (jalankan sekali):
```bash
sudo cp /home/user/cache/nginx/3-demo-full.conf \
         /etc/nginx/sites-available/cache-demo-full
sudo ln -sf /etc/nginx/sites-available/cache-demo-full \
             /etc/nginx/sites-enabled/
sudo mkdir -p /var/cache/nginx/demo_cache
sudo chown www-data:www-data /var/cache/nginx/demo_cache
sudo nginx -t && sudo systemctl reload nginx
```

**URL yang tersedia:**

| Endpoint | Perilaku Cache |
|----------|---------------|
| `http://localhost:8090/` | Demo UI interaktif |
| `http://localhost:8090/no-cache/data.json` | `Cache-Control: no-store` |
| `http://localhost:8090/browser-cache/data.json` | `Cache-Control: max-age=30` |
| `http://localhost:8090/browser-cache/style.css` | `Cache-Control: max-age=30` |
| `http://localhost:8090/proxy-cache/data.json` | Nginx proxy cache 20 detik |
| `http://localhost:8090/purge/data.json` | Bypass & refresh cache |


---
## Bagian 0 — Persiapan

In [ ]:
import requests
import time
import json

# URL demo — semua melalui nginx, tidak ada Python backend
NGINX = "http://localhost:8090"

def print_response(r: requests.Response, label: str = ""):
    """Cetak status, header cache, dan body respons."""
    width = 56
    print("=" * width)
    if label:
        print(f"  {label}")
        print("=" * width)
    print(f"  URL    : {r.url}")
    print(f"  Status : {r.status_code} {r.reason}")
    print(f"  Waktu  : {r.elapsed.total_seconds()*1000:.1f} ms")
    print()
    print("  ── Header Cache ──────────────────────────────")
    CACHE_HEADERS = [
        "cache-control", "expires", "etag", "last-modified",
        "age", "pragma", "vary",
        "x-cache-status", "x-cache-rule", "x-served-from",
    ]
    found = False
    for h in CACHE_HEADERS:
        if h in r.headers:
            print(f"  {h:<26}: {r.headers[h]}")
            found = True
    if not found:
        print("  (tidak ada header cache)")
    if r.headers.get("content-type","").startswith("application/json") and r.status_code != 304:
        print()
        print("  ── Body ──────────────────────────────────────")
        try:
            for line in json.dumps(r.json(), ensure_ascii=False, indent=2).splitlines():
                print(f"  {line}")
        except Exception:
            print(f"  {r.text[:300]}")
    print()

# Cek koneksi nginx
try:
    r = requests.get(f"{NGINX}/", timeout=3)
    print(f"✅  Nginx aktif di {NGINX}  (HTTP {r.status_code})")
    print(f"    Endpoint siap digunakan.")
except Exception as e:
    print(f"❌  Nginx tidak aktif — jalankan perintah aktivasi di atas!\n    Error: {e}")


---
## Bagian 1 — Mengapa Cache Diperlukan?

### Masalah Tanpa Cache

```
Browser ──► Nginx ──► Backend ──► Database
   ▲                                 │
   └───────── respons ───────────────┘
```

Setiap request melewati seluruh rantai. Jika 1.000 pengguna
membuka halaman yang sama secara bersamaan → backend dan
database mendapat 1.000 query identik.

### Solusi dengan Cache

```
Browser ──► Nginx ──[HIT]──► Cache storage
              │
           [MISS] ──► Backend ──► Database
                         │
              simpan ke cache ◄──┘
```

Hanya request **pertama** yang diteruskan ke backend. Request
berikutnya dilayani dari cache.

In [ ]:
# Demonstrasi: no-cache vs proxy-cache — mana yang lebih cepat pada request berulang?
print("Mengirim 5 request ke masing-masing endpoint ...\n")

results = []
for i in range(5):
    # ① no-cache: nginx baca file dari disk setiap kali
    t0 = time.perf_counter()
    r_no = requests.get(f"{NGINX}/no-cache/data.json")
    ms_no = (time.perf_counter() - t0) * 1000

    # ② proxy-cache: nginx ambil dari cache (setelah request pertama)
    t0 = time.perf_counter()
    r_pc = requests.get(f"{NGINX}/proxy-cache/data.json")
    ms_pc = (time.perf_counter() - t0) * 1000

    cs = r_pc.headers.get("x-cache-status", "—")
    print(f"  Req {i+1}: no-cache={ms_no:6.1f}ms  |  "
          f"proxy-cache={ms_pc:6.1f}ms  X-Cache-Status={cs}")
    time.sleep(0.2)

print()
print("Pengamatan:")
print("  - no-cache : waktu relatif konsisten (selalu ke nginx disk)")
print("  - proxy-cache req 1 : MISS (ambil dari origin port 8889)")
print("  - proxy-cache req 2+: HIT  (ambil dari memory/disk cache nginx)")


---
## Bagian 2 — Jenis-Jenis Cache HTTP

| Jenis | Disimpan Di | Siapa yang Melihat | Contoh |
|-------|------------|-------------------|--------|
| **Browser Cache** | Disk/memori browser | Hanya user tersebut | Chrome DevTools → Network → disk cache |
| **Proxy Cache** | Server nginx/Varnish | Semua user | `proxy_cache` di nginx |
| **CDN Cache** | Edge server global | Semua user di region | Cloudflare, Fastly |

### Header HTTP yang Mengontrol Cache

| Header | Arah | Fungsi |
|--------|------|--------|
| `Cache-Control` | Response | Aturan cache utama (max-age, no-store, dll.) |
| `Expires` | Response | Tanggal kedaluwarsa absolut (legacy) |
| `ETag` | Response | Versi konten — hash unik |
| `Last-Modified` | Response | Timestamp terakhir berubah |
| `If-None-Match` | Request | Kirim ETag lama → tanya apakah masih valid |
| `If-Modified-Since` | Request | Kirim tanggal → tanya apakah ada yang baru |
| `Age` | Response | Berapa detik konten sudah ada di cache |
| `Vary` | Response | Cache berbeda berdasarkan header request ini |

In [ ]:
import subprocess, os

print("=" * 60)
print("  Verifikasi Infrastruktur Demo")
print("=" * 60)

checks = [
    ("nginx berjalan?",         ["systemctl", "is-active", "nginx"]),
    ("port 8090 aktif?",        ["bash", "-c", "ss -tlnp | grep :8090"]),
    ("port 8889 aktif?",        ["bash", "-c", "ss -tlnp | grep :8889"]),
    ("file origin ada?",        ["test", "-f", "/var/www/html/cache-demo/origin/data.json"]),
    ("file browser-cache ada?", ["test", "-f", "/var/www/html/cache-demo/assets/data.json"]),
    ("direktori cache ada?",    ["test", "-d", "/var/cache/nginx/demo_cache"]),
]

for label, cmd in checks:
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=3)
        ok = result.returncode == 0 and (result.stdout.strip() != "" or cmd[0] == "test")
        status = "✅" if ok else "❌"
        detail = result.stdout.strip()[:50] if result.stdout.strip() else "ok"
        print(f"  {status}  {label:<30}  {detail}")
    except Exception as e:
        print(f"  ❓  {label:<30}  {e}")

print()
print("Arsitektur:")
print("""
  Browser/Python
       │
       ▼
  Nginx port 8090  ──[/no-cache/]──────► /var/www/html/cache-demo/origin/  (no-store)
       │           ──[/browser-cache/]──► /var/www/html/cache-demo/assets/  (max-age=30)
       │           ──[/proxy-cache/]──┐
       │                              │ proxy_pass + proxy_cache
       │                              ▼
       │                         Nginx port 8889 (origin internal)
       │                              │
       │                              └──► /var/www/html/cache-demo/origin/
       │
       └── Cache hits disimpan di: /var/cache/nginx/demo_cache/
""")


---
## Bagian 3 — Eksperimen: Cache-Control Directives

### 3.1  `no-store` — Sama Sekali Tidak Boleh Disimpan

In [ ]:
print("Kirim 3 request ke /no-cache/data.json (Cache-Control: no-store)")
print()

# Bersihkan proxy cache dulu (kalau ada)
requests.get(f"{NGINX}/purge/data.json")

for i in range(3):
    r = requests.get(f"{NGINX}/no-cache/data.json")
    cc    = r.headers.get("cache-control", "—")
    served = r.headers.get("x-served-from", "—")
    print(f"  Request {i+1}: cache-control='{cc}'  x-served-from='{served}'")

print()
print("Pengamatan: nginx selalu membaca file dari disk.")
print("  Tidak ada 'age' header, tidak ada X-Cache-Status.")
print("  Setiap request = 1 disk read oleh nginx.")


### 3.2  `max-age` — Cache Selama N Detik

In [ ]:
print("Kirim 4 request ke /browser-cache/data.json (Cache-Control: public, max-age=30)")
print("Catatan: header ini HANYA memberi tahu browser untuk cache.")
print("         requests (Python) tidak mematuhi browser cache → setiap request tetap ke nginx.")
print()

for i in range(4):
    r = requests.get(f"{NGINX}/browser-cache/data.json")
    cc    = r.headers.get("cache-control", "—")
    etag  = r.headers.get("etag", "—")
    print(f"  Req {i+1}: cache-control='{cc}'  etag={etag}")
    time.sleep(0.3)

print()
print("Kunci pembelajaran:")
print("  - Header 'Cache-Control: public, max-age=30' dikirim ke BROWSER")
print("  - Browser yang 'pintar' tidak akan kirim request ulang selama 30 detik")
print("  - Di DevTools Chrome: Network → refresh → lihat '(memory cache)' atau '(disk cache)'")
print("  - Python requests tidak mematuhi cache header → selalu kirim request baru")


---
## Bagian 4 — Eksperimen: ETag & Conditional Request

ETag adalah **sidik jari** (fingerprint) konten. Nginx static server otomatis menghasilkan ETag.

```
1. Browser  → GET /browser-cache/data.json
   Nginx    ← 200 OK  ETag: "69b1fd25-21d"
   Browser menyimpan ETag bersama konten di cache.

2. Browser  → GET /browser-cache/data.json  +  If-None-Match: "69b1fd25-21d"
   Nginx memeriksa: ETag cocok dengan file di disk? Ya.
   Nginx    ← 304 Not Modified  (tanpa body → hemat bandwidth!)
   Browser menggunakan konten dari cache-nya sendiri.
```


In [ ]:
# /browser-cache/data.json mengirim ETag karena nginx dikonfigurasi 'etag on'
print("=== Request 1: Tanpa If-None-Match ===")
r1 = requests.get(f"{NGINX}/browser-cache/data.json")
print_response(r1, "GET /browser-cache/data.json (pertama kali)")

etag = r1.headers.get("etag", "")
print(f"  ETag diterima: {etag}")
print()


In [ ]:
print("=== Request 2: Dengan If-None-Match (simulasi browser yang sudah punya ETag) ===")
r2 = requests.get(
    f"{NGINX}/browser-cache/data.json",
    headers={"If-None-Match": etag}
)
print(f"  Status : {r2.status_code} {r2.reason}")
print(f"  Body   : '{r2.text[:60]}' (panjang={len(r2.content)} byte)")
print()

if r2.status_code == 304:
    print("✅  304 Not Modified diterima!")
    print("    → Nginx mengkonfirmasi konten tidak berubah.")
    print("    → Tidak ada body yang dikirim → hemat bandwidth!")
    print("    → Browser menggunakan konten dari cache-nya sendiri.")
elif r2.status_code == 200:
    print("ℹ️  200 OK — ETag tidak dikenali atau berubah (file mungkin berubah).")
    new_etag = r2.headers.get("etag", "")
    print(f"    ETag baru: {new_etag}")


---
## Bagian 5 — Eksperimen: Last-Modified

Mekanisme serupa ETag tapi menggunakan **timestamp** alih-alih hash.
Nginx static server otomatis mengisi `Last-Modified` dari waktu modifikasi file di disk.

```
1. Browser → GET /browser-cache/data.json
   Nginx   ← 200 OK  Last-Modified: Wed, 11 Mar 2026 23:38:41 GMT

2. Browser → GET /browser-cache/data.json
             +  If-Modified-Since: Wed, 11 Mar 2026 23:38:41 GMT
   Nginx memeriksa: apakah file diubah setelah tanggal itu? Tidak.
   Nginx   ← 304 Not Modified  (tanpa body)
```


In [ ]:
# Nginx static file serving secara otomatis mengirim Last-Modified
r1 = requests.get(f"{NGINX}/browser-cache/data.json")
last_mod = r1.headers.get("last-modified", "")
print(f"Last-Modified diterima: {last_mod}")
print(f"Status: {r1.status_code}")
print()

# Request kedua dengan If-Modified-Since
r2 = requests.get(
    f"{NGINX}/browser-cache/data.json",
    headers={"If-Modified-Since": last_mod}
)
print(f"Request kedua (If-Modified-Since: {last_mod})")
print(f"Status: {r2.status_code} — Body length: {len(r2.content)} byte")
print()

if r2.status_code == 304:
    print("✅  304 Not Modified — file tidak berubah sejak tanggal tersebut, hemat bandwidth.")
else:
    print(f"ℹ️  HTTP {r2.status_code} — file dianggap berubah.")


---
## Bagian 6 — Nginx Proxy Cache: MISS → HIT → EXPIRED

Endpoint `/proxy-cache/data.json` menggunakan **nginx proxy cache** internal:
- Nginx port **8090** menerima request dari browser/Python
- Jika cache kosong (MISS) → nginx ambil dari origin port **8889**
- Nginx simpan respons di `/var/cache/nginx/demo_cache/`
- Request berikutnya → HIT (tidak ke port 8889 sama sekali)

### Siklus Hidup Cache

```
MISS ──► (simpan) ──► HIT ──► HIT ──► ... ──► EXPIRED (20s) ──► MISS
  │                                                                │
  └─────────── request ke origin port 8889 ──────────────────────┘
```

| `X-Cache-Status` | Artinya |
|-----------------|---------|
| `MISS`    | Tidak ada di cache → nginx ambil dari origin |
| `HIT`     | Disajikan dari cache di disk nginx |
| `EXPIRED` | Ada di cache tapi 20 detik berlalu → refresh dari origin |
| `STALE`   | Origin error, sajikan data lama |
| `BYPASS`  | Cache sengaja dilewati (`/purge/`) |

**Bersihkan cache sebelum mulai:**
```bash
sudo rm -rf /var/cache/nginx/demo_cache/*
sudo nginx -s reload
```


In [ ]:
# Bersihkan proxy cache dulu via endpoint /purge/
requests.get(f"{NGINX}/purge/data.json")
print("Cache dibersihkan. Mengirim 6 request ke /proxy-cache/data.json ...")
print()
print(f"  {'#':<4} {'X-Cache-Status':>14}  {'Age':>5}  {'Waktu(ms)':>10}")
print(f"  {'─'*44}")

for i in range(6):
    t0 = time.perf_counter()
    r  = requests.get(f"{NGINX}/proxy-cache/data.json")
    dt = (time.perf_counter() - t0) * 1000

    cache_stat = r.headers.get("x-cache-status", "—")
    age        = r.headers.get("age", "—")

    indicator = "← origin dipanggil" if cache_stat == "MISS" else "← dari cache nginx"
    print(f"  {i+1:<4} {cache_stat:>14}  {str(age):>5}  {dt:>10.1f}  {indicator}")
    time.sleep(0.3)

print()
print("Interpretasi:")
print("  • Req 1 → MISS  : cache kosong, nginx ambil dari origin (port 8889)")
print("  • Req 2+ → HIT  : disajikan dari /var/cache/nginx/demo_cache/")
print("  • Setelah 20 detik: EXPIRED → nginx ke origin lagi")
print()
print("Coba inspeksi file cache di disk:")
print("  ls -lh /var/cache/nginx/demo_cache/")


---
## Bagian 7 — Mengukur Manfaat Cache: Benchmark Sederhana

In [ ]:
import statistics

def benchmark(url: str, n: int = 15, label: str = "") -> dict:
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        requests.get(url)
        times.append((time.perf_counter() - t0) * 1000)
    return {
        "label": label, "n": n,
        "mean_ms":   statistics.mean(times),
        "median_ms": statistics.median(times),
        "stdev_ms":  statistics.stdev(times),
        "min_ms":    min(times),
        "max_ms":    max(times),
    }

print("Benchmark 15 request ke masing-masing endpoint:")
print("  A) /no-cache/data.json   — selalu baca dari disk nginx (no cache)")
print("  B) /proxy-cache/data.json — cache nginx aktif (HIT)")
print()

# Pastikan proxy cache terisi dulu (1 MISS dulu)
requests.get(f"{NGINX}/purge/data.json")   # kosongkan
requests.get(f"{NGINX}/proxy-cache/data.json")  # isi (MISS pertama)

res_no    = benchmark(f"{NGINX}/no-cache/data.json",    n=15, label="no-cache (disk read)")
res_cache = benchmark(f"{NGINX}/proxy-cache/data.json", n=15, label="proxy-cache (nginx HIT)")

for res in [res_no, res_cache]:
    print(f"  {res['label']}")
    print(f"    Mean   : {res['mean_ms']:7.2f} ms")
    print(f"    Median : {res['median_ms']:7.2f} ms")
    print(f"    Min    : {res['min_ms']:7.2f} ms")
    print(f"    Max    : {res['max_ms']:7.2f} ms")
    print()

if res_no['mean_ms'] > 0 and res_cache['mean_ms'] > 0:
    speedup = res_no['mean_ms'] / res_cache['mean_ms']
    print(f"  Proxy cache {speedup:.1f}x lebih cepat dari no-cache")
    if speedup < 1.5:
        print("  (Perbedaan kecil — kedua endpoint sama-sama memori nginx, bukan koneksi jauh)")


---
## Bagian 8 — Cache Busting: Memaksa Cache Kedaluwarsa

Masalah umum: cache menyimpan konten lama meskipun backend sudah diperbarui.
Solusi **cache busting**:

| Teknik | Cara |
|--------|------|
| **Query string** | `/style.css?v=2.1.0` |
| **Content hash** | `/style.a3f9b1.css` |
| **Purge via nginx** | `proxy_cache_purge` (modul tambahan) |
| **Bersihkan manual** | `rm -rf /var/cache/nginx/cache_demo/*` |

In [ ]:
# Simulasi cache busting dengan query string berbeda
# URL berbeda = kunci cache berbeda = setiap versi baru MISS

print("Demonstrasi: URL berbeda → kunci cache berbeda → selalu MISS untuk versi baru")
print()

for v in ["v1", "v2", "v3", "v1"]:
    url = f"{NGINX}/proxy-cache/data.json?version={v}"
    r   = requests.get(url)
    cs  = r.headers.get("x-cache-status", "—")
    print(f"  GET /proxy-cache/data.json?version={v}  →  X-Cache-Status={cs}")
    time.sleep(0.2)

print()
print("Perhatikan:")
print("  - v1 request pertama  → MISS  (baru masuk cache)")
print("  - v2, v3              → MISS  (kunci cache berbeda)")
print("  - v1 request kedua    → HIT   (sudah ada di cache)")
print()
print("Ini adalah teknik 'cache busting via query string'.")
print("Di production: /style.css?v=2.0.1 → paksa download ulang file baru")


---
## Bagian 9 — Inspeksi File Cache di Disk

In [ ]:
import subprocess

# Path cache sesuai konfigurasi 3-demo-full.conf
CACHE_DIR = "/var/cache/nginx/demo_cache"

result = subprocess.run(
    ["find", CACHE_DIR, "-type", "f"],
    capture_output=True, text=True
)

if result.returncode == 0 and result.stdout.strip():
    files = result.stdout.strip().splitlines()
    print(f"File cache di disk ({len(files)} file):")
    for f in files[:10]:
        print(f"  {f}")
    if len(files) > 10:
        print(f"  ... dan {len(files)-10} file lainnya")
else:
    print(f"Direktori cache kosong atau belum ada: {CACHE_DIR}")
    print("  → Kirim beberapa request ke /proxy-cache/ dulu, lalu jalankan ulang cell ini.")
    print(f"  → Pastikan nginx sudah di-reload dengan konfigurasi 3-demo-full.conf")


In [ ]:
# Lihat isi salah satu file cache:
# Struktur file: [metadata binary] + [respons HTTP lengkap]
if result.returncode == 0 and result.stdout.strip():
    first_file = result.stdout.strip().splitlines()[0]
    print(f"Isi file: {first_file}")
    print("─" * 50)
    with open(first_file, "rb") as f:
        raw = f.read(900)
    # Decode bagian yang bisa dibaca (metadata binary dilewati nginx internaly)
    text = raw.decode("utf-8", errors="replace")
    print(text)
    print()
    print("Keterangan:")
    print("  - Baris pertama berisi metadata cache nginx (binary + header)")
    print("  - Di bawahnya adalah respons HTTP asli dari origin (port 8889)")
    print("  - Inilah yang dikembalikan nginx saat X-Cache-Status = HIT")
else:
    print("Tidak ada file cache untuk dibaca.")
    print("  Jalankan dulu cell di atas untuk mengisi cache, lalu coba lagi.")


---
## Bagian 10 — Tugas Mandiri

### Tugas A — Analisis Header
Gunakan `print_response()` untuk mengisi tabel berikut:

| Endpoint | Cache-Control | ETag? | X-Cache-Status | Status ke-2 |
|----------|--------------|-------|----------------|-------------|
| `/no-cache/data.json` | | | | |
| `/browser-cache/data.json` | | | | |
| `/proxy-cache/data.json` | | | | |

### Tugas B — Ubah Durasi Proxy Cache
Edit `/home/momotaro/cache/nginx/3-demo-full.conf`:
1. Ganti `proxy_cache_valid 200 20s` menjadi **10 detik**
2. Reload nginx, tunggu 10 detik, kirim request ke `/proxy-cache/` — apakah status menjadi `EXPIRED`?
3. Tambahkan header `X-Lab-Student: NamaMahasiswa` pada semua respons

```bash
sudo cp /home/momotaro/cache/nginx/3-demo-full.conf \
        /etc/nginx/sites-available/cache-demo-full
sudo nginx -t && sudo systemctl reload nginx
```

### Tugas C — Cache Invalidation
1. Kirim beberapa request ke `/proxy-cache/data.json` sampai status `HIT`
2. Edit isi file origin: 
   ```bash
   sudo nano /var/www/html/cache-demo/origin/data.json
   # ubah nilai "versi_konten" dari "1.0" menjadi "2.0"
   ```
3. Kirim request lagi **sebelum** cache expired — apakah konten yang dikembalikan versi baru atau lama?
4. Jelaskan mengapa!


In [ ]:
# ─── Tugas C — Cache Invalidation: isi kode di sini ─────────
# Langkah:
# 1. Kirim request sampai HIT
# 2. Edit /var/www/html/cache-demo/origin/data.json
# 3. Kirim lagi — catat X-Cache-Status dan versi_konten yang dikembalikan

# [MULAI KODE ANDA]

ENDPOINT = f"{NGINX}/proxy-cache/data.json"

# Langkah A: warming cache
r_warm = requests.get(ENDPOINT)
print(f"Warming: {r_warm.headers.get('x-cache-status', '—')}")

# Langkah B: edit file origin via terminal, lalu jalankan cell ini:
# sudo nano /var/www/html/cache-demo/origin/data.json

# Langkah C: kirim setelah edit
r_after = requests.get(ENDPOINT)
print(f"Setelah edit: status={r_after.headers.get('x-cache-status','—')}")
try:
    versi = r_after.json().get("versi_konten", "?")
    print(f"  versi_konten = {versi}")
    print("  (jika masih '1.0' → cache belum expired, konten lama masih aktif)")
except Exception:
    pass

# [AKHIR KODE ANDA]


---
## Bagian 11 — Ringkasan & Poin Kunci

```
┌─────────────────────────────────────────────────────────────┐
│                   Browser Cache vs Proxy Cache              │
├────────────────────┬────────────────────────────────────────┤
│  Browser Cache     │  Proxy Cache (nginx)                   │
├────────────────────┼────────────────────────────────────────┤
│  Lokasi: klien     │  Lokasi: server nginx                  │
│  Scope: 1 user     │  Scope: semua user                     │
│  Header: max-age   │  Directive: proxy_cache_valid          │
│  Kontrol: browser  │  Kontrol: admin server                 │
│  Status: (DevTools)│  Status: X-Cache-Status: HIT/MISS      │
├────────────────────┴────────────────────────────────────────┤
│  Endpoint Demo (port 8090)                                  │
│  /no-cache/          → Cache-Control: no-store              │
│  /browser-cache/     → Cache-Control: max-age=30            │
│  /proxy-cache/       → nginx cache server-side 20s          │
│  /purge/             → bypass & refresh cache               │
├──────────────────────────────────────────────────────────────┤
│  Kapan TIDAK boleh cache?                                    │
│  • Data real-time (harga saham, sensor)                     │
│  • Autentikasi & sesi pengguna                              │
│  • Transaksi keuangan                                        │
│  • Request POST / PUT / DELETE                              │
└──────────────────────────────────────────────────────────────┘
```

### UI Interaktif
Buka browser → `http://localhost:8090/` untuk demo visual browser vs proxy cache.

### Referensi
- [MDN HTTP Caching](https://developer.mozilla.org/en-US/docs/Web/HTTP/Caching)
- [Nginx proxy_cache documentation](https://nginx.org/en/docs/http/ngx_http_proxy_module.html#proxy_cache)
- [RFC 9111 — HTTP Caching](https://www.rfc-editor.org/rfc/rfc9111)
